# **Drug Names: Nativized vs Original Spelling**

The drug-name pipeline turns drug names into IPA with the G2P and then compares the pronunciations.
A drug name can reach it in two spellings: the **original** one, the international generic name
or the brand name as printed (`paracetamol`, `biogesic`), and a **nativized** Filipino respelling
that writes the name as it is said (`parasetamol`, `biyodyesik`). This notebook checks how similar
the G2P's IPA output is for the two spellings of the same drug, and where the two differ.

The G2P here is the WFST model of `Wik_eval.ipynb` (`train/cwik_model.fst`), the model the
command-line tool uses. This check is separate from the choice of G2P in `G2P_compare.ipynb`: no
setting is chosen with it.

There are no reference pronunciations of these drug names, so the check measures how much the two
outputs **agree**, not how accurate either one is. Because the WFST was trained on Filipino
spellings and the nativized spelling writes the pronunciation out, the nativized output is the
point of comparison: an edit is a place where the original spelling makes the model say something
else.

Run `Wik_eval.ipynb` first: it trains the model. The notebook runs in the container, where
Phonetisaurus is installed, and saves both outputs to `output/drug_wfst_output.tsv`.

**The process at a glance**

1. **Load** the 40 drug names in both spellings.
2. **Check** that the model knows every letter of both spellings.
3. **Transcribe** both spellings with the WFST.
4. **Measure** how similar the two outputs are.
5. **Find** where the outputs differ.
6. **The obvious difference:** the silent final *-e*.
7. **Check** which other English spellings change the output.
8. **What this shows and doesn't.**

In [1]:
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import collect_errors
from alignment import align_trxn

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
train_dir   = current_dir / "train"
wlist_dir   = current_dir / "wlist"
output_dir  = current_dir / "output"
output_dir.mkdir(exist_ok=True)

cwik_model = train_dir / "cwik_model.fst"
if not cwik_model.is_file():
    raise FileNotFoundError(f"{cwik_model} not found. Run Wik_eval.ipynb first: it trains the model.")
print("Model:", cwik_model)

Model: /workspaces/Taglog-G2P/notebook/train/cwik_model.fst


## 1. Load the drug names

`wlist/drug_test.wlist` holds 40 nativized drug names. `wlist/drug_pairs.tsv` pairs each one with
its original spelling and a kind: a generic name, a brand name, a herbal medicine, or `other` for
`salmonela`, which is in the word list but names a bacterium, not a drug. For *mefenamic acid*,
*ascorbic acid* and *ferrous sulfate*, only the first word is in the list.

**10** names are spelled the same in both forms: 8 generic names that already follow Filipino
spelling (`losartan`, `metformin`, `ibuprofen`) and the 2 herbal medicines, `lagundi` and
`sambong`. They are a control: the WFST is deterministic, so their two outputs must be identical.
The other **30** are spelled differently, including all 8 brand names.

In [2]:
pairs = pd.read_csv(wlist_dir / "drug_pairs.tsv", sep="\t", dtype=str)

# The nativized column is the existing drug word list, in the same order.
drug_wlist = (wlist_dir / "drug_test.wlist").read_text(encoding="utf-8").split()
assert list(pairs["nativized"]) == drug_wlist, "drug_pairs.tsv and drug_test.wlist don't match"

pairs["same_spelling"] = pairs["nativized"] == pairs["original"]
print(f"Drug names: {len(pairs)}")
print(f"  spelled the same in both   : {pairs['same_spelling'].sum()}")
print(f"  spelled differently        : {(~pairs['same_spelling']).sum()}")
print()
print(pairs.groupby("kind")["same_spelling"].agg(names="size", same_spelling="sum"))

Drug names: 40
  spelled the same in both   : 10
  spelled differently        : 30

         names  same_spelling
kind                         
brand        8              0
generic     29              8
herbal       2              2
other        1              0


## 2. Check the alphabet

Phonetisaurus silently drops a letter it never saw in training, which would make an output
unreliable. Every letter of both spellings is in the model's training words, so nothing is dropped.

The letters that English spellings need are rare in the training data, though: in 19,539 training
words the model saw `c` 56 times, `z` 7 times and `x` and `q` 4 times each. How it reads them
rests on very few examples.

In [3]:
# The model's input alphabet is the set of letters in its training words (cwik_trainval.dict).
train_counts = {}
with open(dicts_dir / "cwik_trainval.dict", encoding="utf-8") as f:
    for line in f:
        for ch in line.split("\t")[0]:
            train_counts[ch] = train_counts.get(ch, 0) + 1

for col in ("nativized", "original"):
    unknown = sorted({ch for w in pairs[col] for ch in w} - set(train_counts))
    print(f"{col:<10} letters the model never saw: {unknown or 'none'}")

print("\nHow often the model saw each English-only letter in training:")
print({ch: train_counts.get(ch, 0) for ch in "cqxjfvz"})

nativized  letters the model never saw: none
original   letters the model never saw: none

How often the model saw each English-only letter in training:
{'c': 56, 'q': 4, 'x': 4, 'j': 26, 'f': 34, 'v': 22, 'z': 7}


## 3. Transcribe both spellings

Both word lists go through `phonetisaurus-apply` with the same model, as in `Wik_eval.ipynb`. Each
word gets its best pronunciation.

In [4]:
def wfst_apply(words):
    """Run phonetisaurus-apply on a word list and return {word: [phones]}."""
    with tempfile.TemporaryDirectory(prefix="drug_") as tmp:
        wlist = Path(tmp) / "drugs.wlist"
        wlist.write_text("\n".join(words), encoding="utf-8")
        out = subprocess.run(
            ["phonetisaurus-apply", "--model", str(cwik_model), "--word_list", str(wlist)],
            capture_output=True, text=True, check=True,
        ).stdout
    hyp = {}
    for line in out.splitlines():
        parts = line.split("\t")
        if len(parts) >= 2:
            hyp[parts[0]] = parts[1].split()
    return hyp


nat_hyp  = wfst_apply(list(pairs["nativized"]))
orig_hyp = wfst_apply(list(dict.fromkeys(pairs["original"])))
assert set(pairs["nativized"]) <= set(nat_hyp) and set(pairs["original"]) <= set(orig_hyp), "a word got no output"

pairs["nativized_phones"] = [nat_hyp[w] for w in pairs["nativized"]]
pairs["original_phones"]  = [orig_hyp[w] for w in pairs["original"]]
for _, r in pairs.head(5).iterrows():
    print(f"  {r.nativized:<14} [{' '.join(r.nativized_phones)}]   {r.original:<14} [{' '.join(r.original_phones)}]")

  parasetamol    [p a ɾ a s e t a m o l]   paracetamol    [p a ɾ a s e t a m o l]
  amoksisilin    [ʔ a m o k s i s i l i n]   amoxicillin    [ʔ a m o k s j s i l l i n]
  mefenamik      [m e f e n a m i k]   mefenamic      [m e f e n a m i k]
  setirizin      [s e t i ɾ i s i n]   cetirizine     [k e t i ɾ i s ʔ i n e ʔ]
  losartan       [l o s a ɾ t a n]   losartan       [l o s a ɾ t a n]


## 4. How similar are the two outputs?

For each drug, the two outputs are aligned with `align_trxn()`, the alignment behind PER in the
other notebooks. The **similarity** is 1 minus the phone edit distance divided by the length of the
longer output: 1 means identical, 0 means nothing in common. The edits are also counted per phone of
the nativized output, which is the PER of the original output if the nativized one is taken as the
reference.

Of the 30 names spelled differently, **13** get exactly the same output (`paracetamol`,
`clonazepam`, `alaxan`, `solmux`) and **17** don't. Their mean similarity is **0.88**, and the
original spelling changes **12.9** phones in every 100. The least similar pairs are `ceelin`
(0.43, `s i l i n` against `k e ʔ e l i n`), `ferrous` (0.57) and `doxycycline` (0.62). The 10
names spelled the same are identical, as expected.

In [5]:
def similarity(a, b):
    """1 - phone edit distance / length of the longer output: 1 is identical, 0 shares nothing."""
    return 1 - align_trxn(a, b)[2] / max(len(a), len(b))


pairs["edits"] = [align_trxn(a, b)[2] for a, b in zip(pairs["nativized_phones"], pairs["original_phones"])]
pairs["similarity"] = [similarity(a, b) for a, b in zip(pairs["nativized_phones"], pairs["original_phones"])]
pairs["identical"] = pairs["edits"] == 0


def summarize(label, sub):
    n_phones = sum(len(p) for p in sub["nativized_phones"])
    return {
        "Drug names": label, "Names": len(sub),
        "Identical outputs": int(sub["identical"].sum()),
        "Mean similarity": round(sub["similarity"].mean(), 3),
        "Min similarity": round(sub["similarity"].min(), 3),
        "Phone edits": int(sub["edits"].sum()),
        "Edits per nativized phone (%)": round(sub["edits"].sum() / n_phones * 100, 2),
    }


display(pd.DataFrame([
    summarize("all", pairs),
    summarize("spelled the same", pairs[pairs["same_spelling"]]),
    summarize("spelled differently", pairs[~pairs["same_spelling"]]),
]))

# Saved for the drug-name pipeline.
saved = pairs.assign(nativized_phones=pairs["nativized_phones"].str.join(" "),
                     original_phones=pairs["original_phones"].str.join(" "),
                     similarity=pairs["similarity"].round(4))
saved[["nativized", "original", "kind", "nativized_phones", "original_phones", "edits", "similarity"]].to_csv(
    output_dir / "drug_wfst_output.tsv", sep="\t", index=False)

table = saved[["nativized", "original", "kind", "nativized_phones", "original_phones", "edits", "similarity"]]
table[~pairs["same_spelling"]].sort_values(["similarity", "nativized"]).reset_index(drop=True)

,Drug names,Names,Identical outputs,Mean similarity,Min similarity,Phone edits,Edits per nativized phone (%)
0,all,40,23,0.91,0.429,37,9.76
1,spelled the same,10,10,1.00,1.000,0,0.00
2,spelled differently,30,13,0.88,0.429,37,12.94


,nativized,original,kind,nativized_phones,original_phones,edits,similarity
0,silin,ceelin,brand,s i l i n,k e ʔ e l i n,4,0.4286
1,feraws,ferrous,generic,f e ɾ a w s,f e ɾ ɾ o u s,3,0.5714
2,doksisayklin,doxycycline,generic,d o k s i s a j k l i n,d o k s j s i k e l i n e,5,0.6154
3,setirizin,cetirizine,generic,s e t i ɾ i s i n,k e t i ɾ i s ʔ i n e ʔ,4,0.6667
4,loperamayd,loperamide,generic,l o p e ɾ a m a j d,l o p e ɾ a m i d e,3,0.7000
5,azitromaysin,azithromycin,generic,ʔ a s i t ɾ o m a j s i n,ʔ a s j t ɾ o m j s j n,3,0.7692
6,biyodyesik,biogesic,brand,b i o d j e s i k,b i o ɡ e s i k,2,0.7778
7,prednisown,prednisone,generic,p ɾ e d n i s o w n,p ɾ e d n i s o n e,2,0.8000
8,amoksisilin,amoxicillin,generic,ʔ a m o k s i s i l i n,ʔ a m o k s j s i l l i n,2,0.8462
9,siprofloksasin,ciprofloxacin,generic,s i p ɾ o f l o k s a s i n,k i p ɾ o f l o k s a s j n,2,0.8571


## 5. Where the outputs differ

The two outputs of each of the 17 names that differ, aligned, with `^` under every difference.
The nativized output is on top. The edit counts below are the operations that turn the nativized
output into the original one.

There are **37** edits: 18 insertions, 14 substitutions and 5 deletions. Read from the
alignments, they come from a few English spelling conventions that the WFST reads with Filipino
letter values:

| Cause | Drug names | What the WFST does |
| --- | --- | --- |
| **Silent final *-e*** | 9: `cetirizine`, `amlodipine`, `omeprazole`, `cotrimoxazole`, `doxycycline`, `prednisone`, `loperamide`, `metronidazole`, `ranitidine` | reads it as a vowel, `/e/` (section 6) |
| ***i* or *y* after *c*, *x*, *z*** | 4: `amoxicillin`, `doxycycline`, `azithromycin` (twice), `ciprofloxacin` | reads the vowel as the glide `j` (`s j n` for *-cin*) |
| **English vowel sounds** | 4: `doxycycline`, `azithromycin`, `loperamide`, `prednisone` | reads *y*, *i–e* and *o–e* as plain vowels, where the nativized spelling writes the English sounds as *ay* and *ow* (`loperamayd`, `prednisown`) |
| **Soft *c* at the start of a word** | 3: `cetirizine`, `ceelin`, `ciprofloxacin` | reads it as `k`; inside a word, as in `paracetamol`, it reads it correctly as `s` |
| **Doubled letters** | 3: `amoxicillin`, `robitussin`, `ferrous` | says the consonant twice (`l l`, `s s`, `ɾ ɾ`); `salmonella` is read with one `l` |
| **Vowel pairs** | 2: `ceelin`, `ferrous` | reads each letter: *ee* as `e ʔ e`, with the glottal stop of two written vowels in Tagalog, and *ou* as `o u` |
| **Soft *g*** | 1: `biogesic` | reads it as `ɡ`, where the nativized spelling writes `dy` (`d j`) |

A name can have more than one cause, so the rows add up to more than 17. Four edits have no
clear cause in the spelling: two glottal stops in `cetirizine`, which go away with its final *-e*
(section 6), a final glottal stop in `salmonella`, and an `e` between `k` and `l` in
`doxycycline`. The glottal stop is also the WFST's most common error on the test split
(`Wik_eval.ipynb`, section 7).

In [6]:
def show_alignment(a, b):
    """The two outputs aligned, with ^ under each position where they differ."""
    aln_a, aln_b, _, _ = align_trxn(a, b)
    width = [max(len(x), len(y)) for x, y in zip(aln_a, aln_b)]
    line = lambda seq: " ".join(s.ljust(w) for s, w in zip(seq, width))
    marks = " ".join(("^" if x != y else " ").ljust(w) for x, y, w in zip(aln_a, aln_b, width))
    return line(aln_a), line(aln_b), marks


differ = pairs[~pairs["identical"]].sort_values("similarity")
print(f"{len(differ)} drug names whose two outputs differ (* = a gap):\n")
for _, r in differ.iterrows():
    a, b, marks = show_alignment(r.nativized_phones, r.original_phones)
    print(f"{r.nativized:<15} {a}")
    print(f"{r.original:<15} {b}")
    print(f"{'':<15} {marks}\n")

17 drug names whose two outputs differ (* = a gap):

silin           * * s i l i n
ceelin          k e ʔ e l i n
                ^ ^ ^ ^      

feraws          f e * ɾ a w s
ferrous         f e ɾ ɾ o u s
                    ^   ^ ^  

doksisayklin    d o k s i s a j k * l i n *
doxycycline     d o k s j s * i k e l i n e
                        ^   ^ ^   ^       ^

setirizin       s e t i ɾ i s * i n * *
cetirizine      k e t i ɾ i s ʔ i n e ʔ
                ^             ^     ^ ^

loperamayd      l o p e ɾ a m a j d *
loperamide      l o p e ɾ a m * i d e
                              ^ ^   ^

azitromaysin    ʔ a s i t ɾ o m a j s i n
azithromycin    ʔ a s j t ɾ o m * j s j n
                      ^         ^     ^  

biyodyesik      b i o d j e s i k
biogesic        b i o * ɡ e s i k
                      ^ ^        

prednisown      p ɾ e d n i s o w n *
prednisone      p ɾ e d n i s o * n e
                                ^   ^

amoksisilin     ʔ a m o k s i s i * l i n
amoxicill

In [7]:
# Edit operations from the nativized output to the original one, keyed by the drug name.
nat_ref  = dict(zip(pairs["nativized"], pairs["nativized_phones"]))
orig_out = dict(zip(pairs["nativized"], pairs["original_phones"]))
err = collect_errors(nat_ref, orig_out)
total = err["n_sub"] + err["n_del"] + err["n_ins"]

display(pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of edits": f"{err['n_sub'] / total * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of edits": f"{err['n_del'] / total * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of edits": f"{err['n_ins'] / total * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total,        "% of edits": "100.0%"},
]))

pd.set_option("display.max_colwidth", 120)
rows = []
for kind, counter, examples in [("inserted", err["ins_counter"], err["ins_examples"]),
                                ("deleted", err["del_counter"], err["del_examples"])]:
    for phone, c in counter.most_common():
        rows.append({"Edit": f"{kind} {phone}", "Count": c,
                     "Drug names": ", ".join(ex["word"] for ex in examples[phone])})
for (r, h), c in err["sub_pair_counter"].most_common():
    rows.append({"Edit": f"{r} -> {h}", "Count": c,
                 "Drug names": ", ".join(ex["word"] for ex in err["sub_pair_examples"][(r, h)])})
pd.DataFrame(rows).sort_values("Count", ascending=False, kind="stable").reset_index(drop=True)

,Type,Count,% of edits
0,Substitution,14,37.8%
1,Deletion,5,13.5%
2,Insertion,18,48.6%
3,TOTAL,37,100.0%


,Edit,Count,Drug names
0,inserted e,11,"setirizin, amlodipin, omeprazol, kotrimoksazol, doksisayklin"
1,i -> j,5,"amoksisilin, doksisayklin, azitromaysin, azitromaysin, siprofloksasin"
2,inserted ʔ,3,"setirizin, setirizin, salmonela"
3,deleted a,3,"doksisayklin, azitromaysin, loperamayd"
4,s -> k,2,"setirizin, siprofloksasin"
5,j -> i,2,"doksisayklin, loperamayd"
6,inserted l,1,amoksisilin
7,inserted k,1,silin
8,inserted s,1,robitusin
9,inserted ɾ,1,feraws


## 6. The obvious difference: the silent final *-e*

English generic names often end in a silent *-e* (*-ine*, *-ole*, *-one*, *-ide*), and the
nativized spelling drops it. Tagalog has no silent letters, so the WFST reads the *e* as a vowel:
`amlodipine` becomes `ʔ a m l o d i p i n e`, a syllable longer than the drug as it is said.

**All 9** original spellings with a silent final *-e* get a final `/e/`. These 9 names are more
than half of the 17 whose outputs differ. To check that the *-e* is the cause, the same 9
spellings are run through the model again without it (`amlodipin`, `cetirizin`):

- **5** become identical to the nativized output: `amlodipine`, `omeprazole`, `cotrimoxazole`,
  `metronidazole` and `ranitidine`. The *-e* was their only difference.
- The edits on the 9 names drop from **19 to 8**, so the *-e* accounts for **11 of the 37** edits,
  more than any other cause. In `cetirizine` it also brings two glottal stops (`s ʔ i n e ʔ`); once
  it is gone, only the soft *c* remains (`k` for `s`).
- The other 4 still differ, for the other causes in section 5 (`doxycycline`, `prednisone`,
  `loperamide` and `cetirizine`).

In [8]:
# An original spelling ending in a silent -e that the nativized spelling drops (amlodipine, amlodipin).
silent_e = pairs[pairs["original"].str.endswith("e") & ~pairs["nativized"].str.endswith("e")].copy()


def last_vowel_e(phones):
    """True if the output ends in /e/, ignoring a final glottal stop (cetirizine: ... n e ʔ)."""
    phones = phones[:-1] if phones[-1] == "ʔ" else phones
    return phones[-1] == "e"


silent_e["final /e/"] = [last_vowel_e(o) and not last_vowel_e(n)
                         for n, o in zip(silent_e["nativized_phones"], silent_e["original_phones"])]

# The same spellings without the -e, through the same model.
no_e = wfst_apply([w[:-1] for w in silent_e["original"]])
silent_e["without -e"] = [w[:-1] for w in silent_e["original"]]
silent_e["edits without -e"] = [align_trxn(n, no_e[w[:-1]])[2]
                                for n, w in zip(silent_e["nativized_phones"], silent_e["original"])]

n_final_e = int(silent_e["final /e/"].sum())
print(f"Original spellings with a silent final -e : {len(silent_e)}")
print(f"  read with a final /e/                   : {n_final_e}")
print(f"  identical to the nativized output once the -e is dropped: "
      f"{int((silent_e['edits without -e'] == 0).sum())}")
print(f"Names whose two outputs differ            : {int((~pairs['identical']).sum())}")
print(f"Phone edits on these names                : {int(silent_e['edits'].sum())} of {int(pairs['edits'].sum())}, "
      f"{int(silent_e['edits without -e'].sum())} once the -e is dropped")
silent_e.assign(nativized_phones=silent_e["nativized_phones"].str.join(" "),
                original_phones=silent_e["original_phones"].str.join(" "))[
    ["nativized", "original", "nativized_phones", "original_phones", "final /e/", "edits",
     "without -e", "edits without -e"]].reset_index(drop=True)

Original spellings with a silent final -e : 9
  read with a final /e/                   : 9
  identical to the nativized output once the -e is dropped: 5
Names whose two outputs differ            : 17
Phone edits on these names                : 19 of 37, 8 once the -e is dropped


,nativized,original,nativized_phones,original_phones,final /e/,edits,without -e,edits without -e
0,setirizin,cetirizine,s e t i ɾ i s i n,k e t i ɾ i s ʔ i n e ʔ,True,4,cetirizin,1
1,amlodipin,amlodipine,ʔ a m l o d i p i n,ʔ a m l o d i p i n e,True,1,amlodipin,0
2,omeprazol,omeprazole,ʔ o m e p ɾ a s o l,ʔ o m e p ɾ a s o l e,True,1,omeprazol,0
3,kotrimoksazol,cotrimoxazole,k o t ɾ i m o k s a s o l,k o t ɾ i m o k s a s o l e,True,1,cotrimoxazol,0
4,doksisayklin,doxycycline,d o k s i s a j k l i n,d o k s j s i k e l i n e,True,5,doxycyclin,4
5,prednisown,prednisone,p ɾ e d n i s o w n,p ɾ e d n i s o n e,True,2,prednison,1
6,loperamayd,loperamide,l o p e ɾ a m a j d,l o p e ɾ a m i d e,True,3,loperamid,2
7,metronidazol,metronidazole,m e t ɾ o n i d a s o l,m e t ɾ o n i d a s o l e,True,1,metronidazol,0
8,ranitidin,ranitidine,ɾ a n i t i d i n,ɾ a n i t i d i n e,True,1,ranitidin,0


## 7. Spellings the WFST reads the nativized way

Each English spelling feature is counted among the 30 names spelled differently, split by whether
the name's two outputs come out identical. The features found in identical pairs are read the
nativized way: the model says them as the Filipino respelling writes them.

- **Hard *c*, *x* and *z*.** A *c* that isn't before *e*, *i* or *y* becomes `k` (`bactidol`,
  `ascorbic`, `clopidogrel`), as the nativized *k* does, and *x* becomes `k s` (`alaxan`,
  `naproxen`), as *ks* does. The nativized spellings write *z* as *s* in two names (`klonasepam`,
  `diyasepam`) but keep it in five (`omeprazol`, `setirizin`); the WFST reads *z* as `s` either
  way.
- ***ia*, *io*.** `diatabs` and `diazepam` are read like `diyatabs` and `diyasepam`: the WFST
  writes no glide for the *y* in either spelling.
- **Soft *c* inside a word.** `paracetamol` is read like `parasetamol`.

None of the names with a silent final *-e*, a doubled letter, an English vowel sound or a soft *g*
comes out identical. The names listed under "Outputs differ" for the other features differ for one
of those causes, not for that feature: every hard *c*, *x*, *z* and *th* in them is read the same
way as in the nativized spelling.

In [9]:
# Spelling features of the original names, split by whether the name's two outputs are identical.
# A name can have several features, so a name listed under "outputs differ" may differ for another one.
FEATURES = {
    "silent final -e":            lambda n, o: o.endswith("e") and not n.endswith("e"),
    "c before e, i, y (soft c)":  lambda n, o: re.search(r"c[eiy]", o) is not None,
    "c elsewhere (hard c)":       lambda n, o: re.search(r"c(?![eiy])", o) is not None,
    "g before e, i (soft g)":     lambda n, o: re.search(r"g[ei]", o) is not None and "dy" in n,
    "x":                          lambda n, o: "x" in o,
    "z":                          lambda n, o: "z" in o,
    "th":                         lambda n, o: "th" in o,
    "doubled letter":             lambda n, o: re.search(r"(.)\1", o) is not None,
    "English vowel (ay, ow in nativized)": lambda n, o: re.search(r"ay|ow", n) is not None and re.search(r"ay|ow", o) is None,
    "ia, io (iya, iyo in nativized)":      lambda n, o: re.search(r"i[ao]", o) is not None and re.search(r"iy[ao]", n) is not None,
}

diff_spelled = pairs[~pairs["same_spelling"]]
rows = []
for label, test in FEATURES.items():
    has = diff_spelled[[test(n, o) for n, o in zip(diff_spelled["nativized"], diff_spelled["original"])]]
    rows.append({
        "Spelling in the original": label, "Names": len(has),
        "Outputs identical": int(has["identical"].sum()),
        "Identical": ", ".join(has.loc[has["identical"], "original"]),
        "Outputs differ": ", ".join(has.loc[~has["identical"], "original"]),
    })
pd.DataFrame(rows)

,Spelling in the original,Names,Outputs identical,Identical,Outputs differ
0,silent final -e,9,0,,"cetirizine, amlodipine, omeprazole, cotrimoxazole, doxycycline, prednisone, loperamide, metronidazole, ranitidine"
1,"c before e, i, y (soft c)",7,1,paracetamol,"amoxicillin, cetirizine, doxycycline, azithromycin, ceelin, ciprofloxacin"
2,c elsewhere (hard c),8,5,"mefenamic, clopidogrel, bactidol, ascorbic, clonazepam","cotrimoxazole, doxycycline, biogesic"
3,"g before e, i (soft g)",1,0,,biogesic
4,x,8,4,"alaxan, solmux, ambroxol, naproxen","amoxicillin, cotrimoxazole, doxycycline, ciprofloxacin"
5,z,7,2,"clonazepam, diazepam","cetirizine, omeprazole, cotrimoxazole, azithromycin, metronidazole"
6,th,1,0,,azithromycin
7,doubled letter,5,0,,"amoxicillin, ceelin, robitussin, ferrous, salmonella"
8,"English vowel (ay, ow in nativized)",4,0,,"doxycycline, azithromycin, prednisone, loperamide"
9,"ia, io (iya, iyo in nativized)",3,2,"diatabs, diazepam",biogesic


## 8. What this shows and doesn't

- **The two spellings give similar outputs, but not the same ones.** Of the 30 names spelled
  differently, 13 get the same IPA, and the mean similarity is 0.88. The rest differ by 1 to 5
  phones, and a pipeline that matches drug names by sound would see the two spellings as
  different words.
- **The obvious difference is the silent final *-e*.** The WFST reads it as `/e/` in all 9 names
  that have it, and it is behind 11 of the 37 edits. It is a systematic effect of English
  spelling, not noise, so original spellings ending in a silent *-e* aren't pronounced the way the
  drug is said.
- **There are no reference pronunciations.** The similarity measures agreement between two
  outputs of the same model. The nativized output is assumed to be the intended pronunciation, but
  it isn't checked against how speakers say the names.
- **One nativized spelling per name.** Each drug has one respelling, from `drug_test.wlist`.
  Another respelling of the same name (*siklin* for *sayklin*) could change the result.
- **The similarity is an unweighted edit distance.** `k` for `s` costs the same as `k` for `a`.
  ALINE, which the pipeline uses to compare drug names, weights a difference by the sounds' phonetic
  features, so its scores will differ. This notebook doesn't run ALINE.
- **Only the WFST.** The fine-tuned CharsiuG2P is tied with the WFST in `G2P_compare.ipynb`. This
  notebook doesn't test whether it reads the original spellings any differently.
- **40 names.** The counts describe this list. They show which spelling conventions cause the
  differences, but not how often each occurs among drug names in general.